# Configure CCM run sets

Also a **utility** notebook: it turns the expanded parameter table into batch job
files — parameter CSVs plus SLURM scripts — for running CCM at scale on an HPC
cluster. Each row becomes one `run_ccm` call per (E, τ, lag, surrogate) combination,
the same operation [A Single CCM Calculation](../2_CCM_single_dyad/1_run__CCMlocally.ipynb) walks through interactively for a
single row. This is the piece that turns the paper's "49 E−τ configurations...
evaluated for over 200 samplings for each of a range of library sizes, for both the
real relationship and again in the context of 400 surrogate relationships" (Results)
from a description into something you can actually submit to a cluster — roughly
49 × 4 dyads × (1 real + 400 surrogate) relationships, each with its own
library-size sweep.


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from operator import index
from pathlib import Path
import os
import sys
import itertools


In [3]:
stem = Path(*(p := Path.cwd().resolve()).parts[: p.parts.index("notebooks")])
proj_stem = stem / 'hol_temp_tsi_ccm'
print(proj_stem)

/Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm


In [7]:
import numpy as np
import pandas as pd
pd.option_context('mode.use_inf_as_na', True)

import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib import gridspec as gs
from matplotlib.markers import MarkerStyle
from matplotlib.lines import Line2D

from collections import defaultdict

import warnings
warnings.filterwarnings("ignore", category=UserWarning)#, module='seaborn')
warnings.filterwarnings("ignore", category=FutureWarning)#, module='seaborn')
warnings.simplefilter("ignore", category=FutureWarning)

from cedarkit.core.project_config import load_config
from cedarkit.utils.cli.arg_parser import get_parser, parse_flags
from cedarkit.utils.routing.paths import *
from cedarkit.utils.routing.file_name_parsers import *
from cedarkit.utils.workflow.slurm_tools import *
from cedarkit.utils.workflow.parameter_utils import make_comb_df


In [5]:
# proj_name, prefix = 'IPSLTR6AVSr02Braconnot19GMSTLinearStationary_Wu18TSILinearStationary', 'ipslGMWuLinStat'
# proj_name = 'GISP2Doering22T15N_Wu18TSI'
# '','', '', '', 'GISP2Alley00TanomLinear_Wu18TSILinear','GISP2Martin24Tanom_Wu18TSI', 'GISP2Martin24TanomLinear_Wu18TSILinear', 'NGRIPMartin24Tanom_Wu18TSI'
# proj_name = 'NGRIPMartin24TanomLinear_Wu18TSILinear'
# proj_name = 'Erb22daGMSTLinear_Wu18TSILinear'
# proj_name = 'GISP2Seierstad14d18O_Wu18TSI'
# proj_name = 'GISP2Doering22T15NLinear_Wu18TSILinear'
# proj_name = 'GISP2Seierstad14d18OLinear_Wu18TSILinear'
# proj_name = 'NGRIP1Seierstad14d18OLinear_Wu18TSILinear'
# proj_name = 'NGRIP1Seierstad14d18O_Wu18TSI'
# proj_name = 'GISP2Alley00Tanom_Wu18TSI'
# proj_name= 'GISP2Alley00Tanom_Vieira11TSI'
dyad_name = 'Erb22daGMST_Wu18TSI'
# proj_name = "Tian22HT115kaALLGMST_Wu18TSI"

# proj_name = 'Erb22daGMST_Vieira11TSI'


In [14]:
dyad_dir = proj_stem/f'{dyad_name}'#Path(os.getcwd()).resolve().parents[0]
config = load_config(dyad_dir / 'proj_config.yaml')
prefix = config.prefix
calc_location =set_calc_path(None, dyad_dir, config, '')
calc_location.mkdir(parents=True, exist_ok=True)
# if (calc_location / 'calc_grps.csv').exists():
#     calc_grps = pd.read_csv(calc_location / 'calc_grps.csv')
# else:
#     calc_grps = pd.DataFrame()

output_location = set_output_path(None, calc_location, config)
output_location.mkdir(parents=True, exist_ok=True)

summary_location = calc_location/'summary_stats'
summary_location.mkdir(parents=True, exist_ok=True)
parameter_dir = dyad_dir / 'parameters'
slurm_dir = dyad_dir / 'slurm'
slurm_dir.mkdir(parents=True, exist_ok=True)

# Figure directories
# figures_dir_parent = proj_dir/'figures'
# figures_dir = figures_dir_parent / notebook_name
# figures_dir.mkdir(exist_ok=True, parents=True)


# Build combination lists

In [9]:
def make_comb_df(col_var_ids=None, target_var_ids=None, E_tau_combs=None, lag_vals=None, tp_vals=None, knn_vals=None, df_path=None):
    if df_path is not None:
        comb_df = pd.read_csv(df_path)
        return comb_df
    else:
        assert col_var_ids is not None
        assert target_var_ids is not None
        assert E_tau_combs is not None
        assert lag_vals is not None
        assert tp_vals is not None
        assert knn_vals is not None
    comb_list = []
    for col_var_id in col_var_ids:
        for target_var_id in target_var_ids:
            for pair in E_tau_combs:
                E, tau = pair
                for lag in lag_vals:
                    for tp in tp_vals:
                        for knn in knn_vals:
                            comb_list.append({
                                'E': E,
                                'tau': tau,
                                'lag': lag,
                                'Tp': tp,
                                'knn': knn,
                                'col_var_id': col_var_id,
                                'target_var_id': target_var_id,
                            })
    comb_df = pd.DataFrame(comb_list)
    return comb_df

# Generate parameter files and slurm scripts

The `surr` branch below switches between the two kinds of batch this pipeline
generates: real relationships (`surr=False`, 200 random library samples per library
size, per Methods) and surrogate relationships (`surr=True`, targeting the 200
phase-randomized replicates per direction from Text S1.3, with a smaller per-sample
count since there are many more surrogate runs to make). `lag_vals` again spans the
±40-decade scan, and `knn_vals=[20]` locks in the "conservative" 20-nearest-neighbor
choice used throughout — instead of the more typical 5 — to cut down on spurious
matches (Text S1.2).


In [15]:
testmode = False
surr = False

E_vals = np.arange(6, 7, 1, dtype=int)
tau_vals = np.arange(1, 7, 1, dtype=int)
E_tau_combs = itertools.product(E_vals, tau_vals)

tp_vals = [1]
knn_vals = [20]
lag_vals = np.arange(-40, 41, 2) #np.arange(0, 6, 1)
models = [config.col.var_id]#['MPI', 'IPSL', 'CCSM3']#, 'MPI']#,'CCSM3',
target_var_ids =[config.target.var_id]#'vieira']

if surr is True:
    groupby_vars = ['tau', 'surr_var', 'E']
    default_calc_length=30
    lag_vals = [0]
    surr_num = 201
    sample = 100
else:
    groupby_vars = ['tau']
    lag_vals = np.arange(-40, 41, 1)
    default_calc_length=15
    surr_num=0
    sample = 200
    
comb_list = [(tau_vals, E_vals, lag_vals)]
comb_df = make_comb_df(models, target_var_ids, E_tau_combs, lag_vals, tp_vals, knn_vals)

gen_parameters_slurm2(dyad_dir, output_location/'parquet', comb_df, parameter_dir=parameter_dir, surr=surr, surr_num=surr_num, groupby_var = groupby_vars, source='parquet', testmode=testmode,config=config,check='consolidated',
                           tp_vals = [1], knn_vals = [20], min_num_to_run=1,
                       # suffix = '_null_check',
                       append=False, proj_prefix= prefix, default_calc_length=default_calc_length,sample=sample,
                                          ntasks=30, max_time_ask=1000, verbose= False)

gen_parameters_slurm2 ['tau']
checking existing outputs in: /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI/calc_local_tmp/calc_refactor/parquet (source=parquet, check=consolidated)
1 could not convert train_len
Generating parameters for slurm scripts: Testmode: False
proj_dir: /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI
output location: /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI/calc_local_tmp/calc_refactor/parquet
	comb_df: (486, 7), surr bool: False, target surr var/num: ['neither']; 0, groupby_var: ['tau']
Slurm scripts will be generated in:, /Users/jlanders/PycharmProjects/hol_temp_tsi_ccm_pb/hol_temp_tsi_ccm/Erb22daGMST_Wu18TSI / slurm
------------------------------------------------------------
gen_slurm_param_from_params ['tau']
combined_df               id  tau  E  train_len  train_ind_i  knn  Tp_flag  Tp  lag  \
0  1786473082781    1  6        Na